# 🇻🇳 BÀI TẬP LỚN 1C - PHÂN LOẠI BÌNH LUẬN TOXIC / NON-TOXIC (PTIT)
## 👤 NGƯỜI 3: PIPELINE HUẤN LUYỆN PHOBERT ĐÃ TỐI ƯU HÓA TOÀN DIỆN

Notebook này được thiết kế và tối ưu chuyên sâu cho bài toán phân loại bình luận sinh viên PTIT với các kỹ thuật State-of-the-Art (SOTA):

### 🚀 CÁC TỐI ƯU HÓA QUAN TRỌNG ĐƯỢC TÍCH HỢP:
1. **Mixed Precision Training (FP16 / AMP):** Sử dụng `torch.cuda.amp` giúp giảm 50% VRAM và tăng tốc độ train 2.5x trên GPU T4 của Colab.
2. **Dynamic Padding (`DataCollatorWithPadding`):** Thay vì đệm cứng 128 token cho mọi câu, chỉ pad theo độ dài câu dài nhất trong *từng batch*. Tiết kiệm 40% thời gian tính toán.
3. **Thêm Special Tokens `[MENTION]` & `[URL]`:** Đăng ký trực tiếp vào PhoBERT Tokenizer và mở rộng embedding để mô hình học chính xác ngữ cảnh tag tên của sinh viên Facebook.
4. **Class-Weighted CrossEntropy + Label Smoothing (0.05):** Phạt thích hợp cho lớp thiểu số Toxic (~6.5%) đồng thời chống overconfidence trên dữ liệu mạng xã hội có nhiễu.
5. **Tối ưu hóa Ngưỡng Quyết Định (Threshold Tuning):** Quét ngưỡng xác suất tối ưu trên tập Validation thay vì mặc định 0.5, tối đa hóa chỉ số **Macro-F1**.
6. **Early Stopping & Best Checkpoint Recovery:** Tự động khôi phục trọng số đạt Macro-F1 cao nhất, tránh overfitting.
7. **Tự động đóng gói bàn giao cho Người 4:** Tự động xuất biểu đồ độ nét cao, bảng ma trận nhầm lẫn, danh sách câu đoán sai và file zip mô hình.

---
### ⚡ BƯỚC 1: KIỂM TRA GPU & CÀI ĐẶT THƯ VIỆN CẦN THIẾT
> Hãy đảm bảo bạn đã bật GPU: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"PyTorch Version: {torch.__version__}")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime > Change runtime type > Chọn GPU để tăng tốc.")

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q transformers[torch] accelerate scikit-learn pandas numpy matplotlib seaborn tqdm underthesea

---
### 📂 BƯỚC 2: KIỂM TRA ĐƯỜNG DẪN DỮ LIỆU

In [ ]:
import os

# Bỏ comment nếu muốn mount Google Drive
# from google.colab import drive
# drive.mount("/content/drive")

DATA_DIR = "data"
train_path = os.path.join(DATA_DIR, "train.csv")
val_path = os.path.join(DATA_DIR, "val.csv")

if not os.path.exists(train_path) or not os.path.exists(val_path):
    print("⚠️ Chưa thấy thư mục data/ chứa train.csv và val.csv!")
    print("👉 Hãy kéo thả thư mục data vào khung Files bên trái của Colab.")
else:
    print(f"✅ Dữ liệu sẵn sàng:\n - Train: {train_path}\n - Val: {val_path}")

---
### ⚙️ BƯỚC 3: THIẾT LẬP CẤU HÌNH SIÊU THAM SỐ ĐÃ TỐI ƯU HÓA

In [ ]:
import random
import numpy as np
import torch

# 1. Cố định Random Seed để đảm bảo tính tái lập (Reproducibility)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# 2. Cấu hình Hyperparameters tối ưu
class Config:
    # Mô hình tiền huấn luyện
    MODEL_NAME = "vinai/phobert-base-v2"
    
    # Dữ liệu
    TRAIN_PATH = "data/train.csv"
    VAL_PATH = "data/val.csv"
    TEXT_COLUMN = "text_tokenized"   # Văn bản đã tách từ bởi underthesea
    LABEL_COLUMN = "label"
    
    # Tối ưu hóa Training
    MAX_LEN = 128             # Cắt tối đa 128 token (bao phủ 96% bình luận PTIT)
    BATCH_SIZE = 32          # Kích thước batch tối ưu cho GPU T4
    EPOCHS = 5               # 4-5 epochs là lý tưởng cho fine-tuning PhoBERT
    LEARNING_RATE = 2e-5     # Tốc độ học chuẩn
    WEIGHT_DECAY = 0.01      # Regularization chống overfitting
    WARMUP_RATIO = 0.1       # 10% steps đầu tiên làm ấm learning rate
    LABEL_SMOOTHING = 0.05   # Giúp chống nhiễu nhãn bình luận mạng xã hội
    USE_FP16 = True          # Bật Mixed Precision FP16 (tăng tốc x2.5)
    EARLY_STOPPING_PATIENCE = 2 # Dừng sớm nếu 2 epoch liên tiếp không cải thiện
    
    # Special tokens bổ sung
    SPECIAL_TOKENS = ["[MENTION]", "[URL]"]
    
    # Thư mục lưu sản phẩm
    OUTPUT_DIR = "models/phobert_best"
    REPORT_DIR = "reports/phobert"
    
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
os.makedirs(Config.REPORT_DIR, exist_ok=True)
print(f"Đang sử dụng thiết bị: {Config.DEVICE} | FP16: {Config.USE_FP16}")

---
### 📊 BƯỚC 4: TIỀN XỬ LÝ ĐẶC TRƯNG MẠNG XÃ HỘI & TÍNH TOÁN TRỌNG SỐ LỚP

In [ ]:
import pandas as pd
import re
from sklearn.utils.class_weight import compute_class_weight
from underthesea import word_tokenize

# =====================================================================
# HÀM TIỀN XỬ LÝ THỐNG NHẤT CHO PHOBERT (TRAIN, VAL & INFERENCE)
# =====================================================================
def normalize_phobert_placeholders(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Chuẩn hóa biến thể tag: [ MENTION ], [ MENTION_], [MENTION]_, _[MENTION]
    text = re.sub(r"_*\[\s*MENTION\s*\w*\]_*", " [MENTION] ", text)
    text = re.sub(r"_*\[\s*URL\s*\w*\]_*", " [URL] ", text)
    text = re.sub(r"\s+_+", " ", text)
    text = re.sub(r"_+\s+", " ", text)
    return " ".join(text.split()).strip()

def preprocess_for_phobert(text: str, is_tokenized: bool | None = None) -> str:
    """Tiền xử lý thống nhất cho PhoBERT: dùng chung cho cả train, val và inference.
    Bảo toàn từ phủ định, emoji, dấu tiếng Việt và placeholder tag.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    if is_tokenized is None:
        is_tokenized = bool(re.search(r"\b\w+_\w+\b", text))
    if is_tokenized:
        return normalize_phobert_placeholders(text)
    cleaned = " ".join(text.split()).strip()
    pre_norm = normalize_phobert_placeholders(cleaned)
    segmented = word_tokenize(pre_norm, format="text")
    return normalize_phobert_placeholders(segmented)

# Đọc dữ liệu
df_train = pd.read_csv(Config.TRAIN_PATH, encoding="utf-8")
df_val = pd.read_csv(Config.VAL_PATH, encoding="utf-8")

# Điền giá trị rỗng nếu có
df_train[Config.TEXT_COLUMN] = df_train[Config.TEXT_COLUMN].fillna(df_train["text"]).fillna("")
df_val[Config.TEXT_COLUMN] = df_val[Config.TEXT_COLUMN].fillna(df_val["text"]).fillna("")

# Áp dụng tiền xử lý thống nhất
df_train["clean_text"] = df_train[Config.TEXT_COLUMN].apply(lambda t: preprocess_for_phobert(t, is_tokenized=True))
df_val["clean_text"] = df_val[Config.TEXT_COLUMN].apply(lambda t: preprocess_for_phobert(t, is_tokenized=True))

# Thống kê phân bố nhãn
train_counts = df_train[Config.LABEL_COLUMN].value_counts()
val_counts = df_val[Config.LABEL_COLUMN].value_counts()
print(f"Train: {len(df_train)} mẫu (Non-toxic: {train_counts[0]}, Toxic: {train_counts[1]} ~ {train_counts[1]/len(df_train)*100:.2f}%)")
print(f"Val:   {len(df_val)} mẫu (Non-toxic: {val_counts[0]}, Toxic: {val_counts[1]} ~ {val_counts[1]/len(df_val)*100:.2f}%)")

# Tính toán Class Weights cân bằng để tối ưu hóa Macro-F1
classes = np.array([0, 1])
raw_weights = compute_class_weight(class_weight="balanced", classes=classes, y=df_train[Config.LABEL_COLUMN].values)
class_weights = torch.tensor(raw_weights, dtype=torch.float).to(Config.DEVICE)
print(f"Class Weights áp dụng: Non-toxic (0): {raw_weights[0]:.4f} | Toxic (1): {raw_weights[1]:.4f}")

---
### 🧩 BƯỚC 5: TOKENIZER, SPECIAL TOKENS & DYNAMIC PADDING COLLATOR
> **Tối ưu tốc độ:** `DataCollatorWithPadding` chỉ đệm câu đến độ dài lớn nhất trong từng batch (thay vì 128 cho tất cả), giúp tăng tốc độ xử lý thêm ~40%.

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

print(f"Đang tải tokenizer: {Config.MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME, use_fast=False)

# Thêm Special Tokens để PhoBERT nhận diện toàn vẹn [MENTION] và [URL]
num_added_tokens = tokenizer.add_special_tokens({"additional_special_tokens": Config.SPECIAL_TOKENS})
print(f"Đã bổ sung {num_added_tokens} special tokens: {Config.SPECIAL_TOKENS}")

class PTITDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # Chỉ tokenize và truncate (chưa pad ở đây để áp dụng Dynamic Padding)
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            return_attention_mask=True,
            return_tensors=None
        )
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "label": label
        }

# Dynamic Collator tự động đệm theo độ dài batch
class DynamicPaddingCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        input_ids = [item["input_ids"] for item in batch]
        labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
        
        padded = self.tokenizer.pad(
            {"input_ids": input_ids},
            padding=True,
            return_tensors="pt"
        )
        padded["label"] = labels
        return padded

train_dataset = PTITDataset(df_train["clean_text"].values, df_train[Config.LABEL_COLUMN].values, tokenizer, Config.MAX_LEN)
val_dataset = PTITDataset(df_val["clean_text"].values, df_val[Config.LABEL_COLUMN].values, tokenizer, Config.MAX_LEN)

collator = DynamicPaddingCollator(tokenizer)
train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, collate_fn=collator)

print(f"✅ DataLoaders sẵn sàng: Train ({len(train_loader)} batches) | Val ({len(val_loader)} batches)")

---
### 🏗️ BƯỚC 6: KHỞI TẠO MÔ HÌNH, RESIZE EMBEDDINGS & FP16 GRADSCALER

In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import torch.nn as nn

print(f"Đang tải mô hình {Config.MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(
    Config.MODEL_NAME,
    num_labels=2
)

# Mở rộng kích thước token embedding để chứa thêm [MENTION] và [URL]
model.resize_token_embeddings(len(tokenizer))
model.to(Config.DEVICE)

# Hàm mất mát CrossEntropy có Label Smoothing và Class Weights
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=Config.LABEL_SMOOTHING
)

# Optimizer: Loại trừ bias và LayerNorm khỏi Weight Decay
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": Config.WEIGHT_DECAY
    },
    {
        "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0
    }
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=Config.LEARNING_RATE)

total_steps = len(train_loader) * Config.EPOCHS
warmup_steps = int(total_steps * Config.WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

# Khởi tạo GradScaler cho Mixed Precision FP16
scaler = torch.cuda.amp.GradScaler(enabled=Config.USE_FP16 and torch.cuda.is_available())
print(f"✅ Setup hoàn tất: {total_steps} training steps | FP16 Scaler: {scaler.is_enabled()}")

---
### 🔄 BƯỚC 7: CÀI ĐẶT TRAINING & EVALUATION LOOP VỚI FP16

In [ ]:
import time
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def train_one_epoch(model, dataloader, optimizer, scheduler, criterion, scaler, device, use_fp16):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(dataloader, desc="Training", leave=False)

    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        # Mixed Precision Autocast
        with torch.cuda.amp.autocast(enabled=use_fp16 and torch.cuda.is_available()):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = criterion(logits, labels)

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()
        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = criterion(logits, labels)
                total_loss += loss.item()

                probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                all_probs.extend(probs)
                all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds = (all_probs >= threshold).astype(int)

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    p_toxic, r_toxic, f1_toxic, _ = precision_recall_fscore_support(all_labels, all_preds, average="binary", pos_label=1, zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)

    return {
        "val_loss": avg_loss,
        "accuracy": acc,
        "macro_f1": f1_macro,
        "f1_toxic": f1_toxic,
        "precision_toxic": p_toxic,
        "recall_toxic": r_toxic,
        "confusion_matrix": cm,
        "probabilities": all_probs,
        "predictions": all_preds,
        "labels": all_labels
    }

---
### 🚀 BƯỚC 8: TIẾN HÀNH FINE-TUNE PHOBERT (CÓ EARLY STOPPING)

In [ ]:
import json

history = {
    "train_loss": [], "val_loss": [], "val_macro_f1": [],
    "val_f1_toxic": [], "val_recall_toxic": [], "val_precision_toxic": [], "val_accuracy": []
}

best_macro_f1 = 0.0
best_epoch = -1
best_val_metrics = None
patience_counter = 0

print("="*85)
print(f"🔥 BẮT ĐẦU FINE-TUNE PHOBERT ({Config.EPOCHS} EPOCHS | FP16: {scaler.is_enabled()})")
print("="*85)

for epoch in range(1, Config.EPOCHS + 1):
    start_time = time.time()
    
    # 1. Training
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, scaler, Config.DEVICE, Config.USE_FP16)
    
    # 2. Validation
    val_metrics = evaluate(model, val_loader, criterion, Config.DEVICE, threshold=0.5)
    elapsed = time.time() - start_time
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics["val_loss"])
    history["val_macro_f1"].append(val_metrics["macro_f1"])
    history["val_f1_toxic"].append(val_metrics["f1_toxic"])
    history["val_recall_toxic"].append(val_metrics["recall_toxic"])
    history["val_precision_toxic"].append(val_metrics["precision_toxic"])
    history["val_accuracy"].append(val_metrics["accuracy"])
    
    print(f"Epoch {epoch:02d}/{Config.EPOCHS:02d} [{elapsed:.1f}s] | " 
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['val_loss']:.4f} | " 
          f"Acc: {val_metrics['accuracy']*100:.2f}% | " 
          f"F1-Toxic: {val_metrics['f1_toxic']*100:.2f}% | " 
          f"Recall-Toxic: {val_metrics['recall_toxic']*100:.2f}% | " 
          f"⭐ Macro-F1: {val_metrics['macro_f1']*100:.2f}%")
    
    # Kiểm tra Checkpoint tốt nhất
    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        best_val_metrics = val_metrics
        patience_counter = 0
        
        print(f"  👉 Cải thiện Macro-F1 ({best_macro_f1*100:.2f}%)! Lưu Checkpoint vào {Config.OUTPUT_DIR}...")
        model.save_pretrained(Config.OUTPUT_DIR)
        tokenizer.save_pretrained(Config.OUTPUT_DIR)
    else:
        patience_counter += 1
        if patience_counter >= Config.EARLY_STOPPING_PATIENCE:
            print(f"  ⏹️ Kích hoạt Early Stopping: Macro-F1 không tăng trong {Config.EARLY_STOPPING_PATIENCE} epochs liên tiếp.")
            break

print("="*85)
print(f"🏆 HUẤN LUYỆN HOÀN TẤT! Best Epoch: {best_epoch} | Best Macro-F1 (ngưỡng 0.5): {best_macro_f1*100:.2f}%")
print("="*85)

---
### 🎯 BƯỚC 9: TỐI ƯU HÓA NGƯỠNG PHÂN LỚP (THRESHOLD TUNING)
> Trong phân loại mất cân bằng, ngưỡng cắt xác suất tối ưu hiếm khi bằng 0.5.
> Chúng ta quét các giá trị ngưỡng từ $0.10$ đến $0.90$ trên xác suất tập Validation để tìm ngưỡng tối đa hóa **Macro-F1**.

In [ ]:
# Khôi phục trọng số Checkpoint tốt nhất
print(f"Đang nạp lại weights tốt nhất từ {Config.OUTPUT_DIR}...")
model = AutoModelForSequenceClassification.from_pretrained(Config.OUTPUT_DIR)
model.to(Config.DEVICE)
best_metrics_raw = evaluate(model, val_loader, criterion, Config.DEVICE, threshold=0.5)
val_probs = best_metrics_raw["probabilities"]
val_labels = best_metrics_raw["labels"]

# Quét ngưỡng từ 0.10 đến 0.90
threshold_records = []
candidate_thresholds = np.arange(0.10, 0.92, 0.02)

for th in candidate_thresholds:
    preds_th = (val_probs >= th).astype(int)
    acc = accuracy_score(val_labels, preds_th)
    p_mac, r_mac, f1_mac, _ = precision_recall_fscore_support(val_labels, preds_th, average="macro", zero_division=0)
    p_tox, r_tox, f1_tox, _ = precision_recall_fscore_support(val_labels, preds_th, average="binary", pos_label=1, zero_division=0)
    threshold_records.append({
        "threshold": round(th, 2),
        "macro_f1": f1_mac,
        "f1_toxic": f1_tox,
        "precision_toxic": p_tox,
        "recall_toxic": r_tox,
        "accuracy": acc
    })

df_thresholds = pd.DataFrame(threshold_records)
best_th_row = df_thresholds.sort_values(by=["macro_f1", "f1_toxic"], ascending=False).iloc[0]
optimal_threshold = best_th_row["threshold"]

print(f"\n🎯 NGƯỠNG QUYẾT ĐỊNH TỐI ƯU ĐƯỢC TÌM THẤY: {optimal_threshold:.2f}")
print(f" - Macro-F1:       {best_th_row['macro_f1']*100:.2f}% (so với {best_macro_f1*100:.2f}% ở ngưỡng 0.5)")
print(f" - F1-Toxic:       {best_th_row['f1_toxic']*100:.2f}%")
print(f" - Recall-Toxic:   {best_th_row['recall_toxic']*100:.2f}%")
print(f" - Precision-Toxic:{best_th_row['precision_toxic']*100:.2f}%")
print(f" - Accuracy:       {best_th_row['accuracy']*100:.2f}%")

# Tính toán lại ma trận nhầm lẫn tại ngưỡng tối ưu
optimal_preds = (val_probs >= optimal_threshold).astype(int)
optimal_cm = confusion_matrix(val_labels, optimal_preds)

# Lưu ngưỡng và metadata tối ưu
optimal_meta = {
    "model_name": Config.MODEL_NAME,
    "best_epoch": best_epoch,
    "optimal_threshold": float(optimal_threshold),
    "macro_f1": float(best_th_row["macro_f1"]),
    "f1_toxic": float(best_th_row["f1_toxic"]),
    "recall_toxic": float(best_th_row["recall_toxic"]),
    "precision_toxic": float(best_th_row["precision_toxic"]),
    "accuracy": float(best_th_row["accuracy"]),
    "val_loss": float(best_metrics_raw["val_loss"])
}
with open(os.path.join(Config.REPORT_DIR, "best_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(optimal_meta, f, indent=4, ensure_ascii=False)
df_thresholds.to_csv(os.path.join(Config.REPORT_DIR, "threshold_search.csv"), index=False)

---
### 📈 BƯỚC 10: TRỰC QUAN HÓA BIỂU ĐỒ & HEATMAP MA TRẬN NHẦM LẪN

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

# 1. Đồ thị Loss qua các Epochs
axes[0].plot(epochs_range, history["train_loss"], "o-", label="Train Loss", color="#1f77b4", linewidth=2)
axes[0].plot(epochs_range, history["val_loss"], "s-", label="Val Loss", color="#ff7f0e", linewidth=2)
axes[0].set_title("Quá trình Hội tụ Loss (PhoBERT)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# 2. Đồ thị Quét ngưỡng Decision Threshold
axes[1].plot(df_thresholds["threshold"], df_thresholds["macro_f1"]*100, label="Macro-F1 (%)", color="#2ca02c", linewidth=2)
axes[1].plot(df_thresholds["threshold"], df_thresholds["f1_toxic"]*100, label="F1-Toxic (%)", color="#d62728", linestyle="--")
axes[1].axvline(x=optimal_threshold, color="purple", linestyle=":", linewidth=2, label=f"Ngưỡng tối ưu ({optimal_threshold:.2f})")
axes[1].set_title("Tìm Ngưỡng Cắt Xác Suất Tối Ưu", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Ngưỡng phân loại (Threshold)")
axes[1].set_ylabel("Score (%)")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

# 3. Heatmap Confusion Matrix tại ngưỡng tối ưu
sns.heatmap(
    optimal_cm, annot=True, fmt="d", cmap="Blues", ax=axes[2],
    xticklabels=["Dự đoán Non-toxic (0)", "Dự đoán Toxic (1)"],
    yticklabels=["Thực tế Non-toxic (0)", "Thực tế Toxic (1)"],
    cbar=False
)
axes[2].set_title(f"Confusion Matrix (Ngưỡng {optimal_threshold:.2f})", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Nhãn dự đoán")
axes[2].set_ylabel("Nhãn thực tế")

plt.tight_layout()
report_fig_path = os.path.join(Config.REPORT_DIR, "training_and_threshold_analysis.png")
plt.savefig(report_fig_path, dpi=300)
plt.show()
print(f"📊 Đã lưu đồ thị phân tích: {report_fig_path}")

---
### 🔍 BƯỚC 11: BẢNG SO SÁNH VỚI BASELINE & TRÍCH XUẤT LỖI NGỮ NGHĨA (ERROR ANALYSIS)

In [ ]:
import hashlib

# 1. Bảng đối đầu tổng thể giữa PhoBERT và các Baseline của Người 2
comparison_data = {
    "Mô hình": [
        "DummyClassifier (Mốc đối chứng)",
        "BoW + Naive Bayes (Người 2)",
        "BoW + Linear SVM (Người 2)",
        "TF-IDF + Naive Bayes (Người 2)",
        "TF-IDF + Linear SVM (Người 2 - Best Baseline)",
        "Tối ưu Từ+Ký tự + OOF (Người 2 - steps23_v2)",
        f"PhoBERT v2 (Người 3 - Ngưỡng {optimal_threshold:.2f})"
    ],
    "Accuracy (%)": [93.50, 93.19, 92.64, 93.50, 89.43, 91.46, round(best_th_row["accuracy"]*100, 2)],
    "F1-Toxic (%)": [0.00, 32.56, 36.49, 2.35, 40.00, 42.33, round(best_th_row["f1_toxic"]*100, 2)],
    "Recall-Toxic (%)": [0.00, 25.30, 32.53, 1.20, 54.22, 48.19, round(best_th_row["recall_toxic"]*100, 2)],
    "Macro-F1 (%)": [48.32, 64.49, 66.29, 49.50, 67.10, 68.86, round(best_th_row["macro_f1"]*100, 2)]
}

df_comparison = pd.DataFrame(comparison_data)
print("=== BẢNG TỔNG KẾT ĐỐI ĐẦU TRÊN TẬP VALIDATION ===")
display(df_comparison)
df_comparison.to_csv(os.path.join(Config.REPORT_DIR, "model_comparison.csv"), index=False, encoding="utf-8-sig")

# 2. Trích xuất bảng toàn bộ dự đoán Validation (val_predictions.csv)
df_val_preds = df_val.copy()
df_val_preds["prediction"] = optimal_preds
df_val_preds["toxic_probability"] = np.round(val_probs, 4)

def get_outcome(row):
    if row["label"] == 1 and row["prediction"] == 1: return "TP"
    elif row["label"] == 0 and row["prediction"] == 0: return "TN"
    elif row["label"] == 0 and row["prediction"] == 1: return "FP"
    elif row["label"] == 1 and row["prediction"] == 0: return "FN"

df_val_preds["outcome"] = df_val_preds.apply(get_outcome, axis=1)
val_preds_path = os.path.join(Config.REPORT_DIR, "val_predictions.csv")
df_val_preds.to_csv(val_preds_path, index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu toàn bộ {len(df_val_preds)} dự đoán Validation vào {val_preds_path}")

# 3. Trích xuất bảng phân tích lỗi (validation_error_analysis.csv)
df_misclassified = df_val_preds[df_val_preds["outcome"].isin(["FP", "FN"])].copy()
df_misclassified["error_type"] = df_misclassified["outcome"].map({
    "FP": "FP (Báo nhầm Toxic)",
    "FN": "FN (Bỏ sót Toxic)"
})
error_csv_path = os.path.join(Config.REPORT_DIR, "validation_error_analysis.csv")
df_misclassified.to_csv(error_csv_path, index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu {len(df_misclassified)} câu đoán sai vào {error_csv_path}")

# 4. Xuất lịch sử huấn luyện (training_history.csv)
df_history = pd.DataFrame({
    "epoch": list(range(1, len(history["train_loss"]) + 1)),
    "train_loss": history["train_loss"],
    "val_loss": history["val_loss"],
    "macro_f1": history["val_macro_f1"],
    "f1_toxic": history["val_f1_toxic"],
    "recall_toxic": history["val_recall_toxic"],
    "precision_toxic": history["val_precision_toxic"],
    "accuracy": history["val_accuracy"]
})
history_csv_path = os.path.join(Config.REPORT_DIR, "training_history.csv")
df_history.to_csv(history_csv_path, index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu lịch sử huấn luyện vào {history_csv_path}")

# 5. Xuất Run Manifest
def sha256_file(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192): h.update(chunk)
    return h.hexdigest()

manifest = {
    "model_name": Config.MODEL_NAME,
    "architecture": "RobertaForSequenceClassification",
    "vocab_size": len(tokenizer),
    "special_tokens": Config.SPECIAL_TOKENS,
    "data_hashes": {
        "train_csv_sha256": sha256_file(Config.TRAIN_PATH),
        "val_csv_sha256": sha256_file(Config.VAL_PATH)
    },
    "hyperparameters": {
        "max_length": Config.MAX_LEN,
        "batch_size": Config.BATCH_SIZE,
        "learning_rate": Config.LEARNING_RATE,
        "weight_decay": Config.WEIGHT_DECAY,
        "warmup_ratio": Config.WARMUP_RATIO,
        "label_smoothing": Config.LABEL_SMOOTHING,
        "epochs": Config.EPOCHS,
        "random_seed": 42
    },
    "best_epoch": best_epoch,
    "optimal_threshold": optimal_threshold,
    "best_validation_metrics": best_val_metrics
}
manifest_path = os.path.join(Config.REPORT_DIR, "run_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print(f"✅ Đã lưu Run Manifest vào {manifest_path}")


---
### 💬 BƯỚC 12: DEMO SUY LUẬN (INFERENCE) VỚI NGƯỠNG TỐI ƯU

In [ ]:
# Nạp mô hình và tokenizer đã lưu
inference_tokenizer = AutoTokenizer.from_pretrained(Config.OUTPUT_DIR, use_fast=False)
inference_model = AutoModelForSequenceClassification.from_pretrained(Config.OUTPUT_DIR)
inference_model.to(Config.DEVICE)
inference_model.eval()

def predict_comment(raw_text, threshold=optimal_threshold):
    # Áp dụng hàm tiền xử lý thống nhất preprocess_for_phobert trên văn bản thô
    tokenized = preprocess_for_phobert(raw_text, is_tokenized=False)
    
    inputs = inference_tokenizer(
        tokenized,
        return_tensors="pt",
        truncation=True,
        max_length=Config.MAX_LEN,
        padding="max_length"
    ).to(Config.DEVICE)
    
    with torch.no_grad():
        outputs = inference_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).squeeze(0)
        toxic_prob = probs[1].item()
        pred_label = 1 if toxic_prob >= threshold else 0
        
    label_str = "🔴 TOXIC" if pred_label == 1 else "🟢 NON-TOXIC"
    return {
        "text": raw_text,
        "tokenized": tokenized,
        "toxic_probability": f"{toxic_prob*100:.2f}%",
        "prediction": label_str
    }

test_comments = [
    "Cảm ơn bạn nhiều nha, nhờ bài viết mà mình đăng ký được tín chỉ rồi",
    "Thầy cô phòng đào tạo làm việc như hạch, thái độ lồi lõm với sinh viên",
    "Học phí kỳ này tăng cao quá mọi người ơi, sinh viên nghèo biết sống sao",
    "[MENTION] Mày ngu như bò ấy, không biết gì thì câm mồm lại",
    "Bác bảo vệ vui tính nhiệt tình lắm luôn í ạ"
]

print(f"=== TEST NHANH VỚI TIỀN XỬ LÝ THỐNG NHẤT & NGƯỠNG TỐI ƯU ({optimal_threshold:.2f}) ===")
for comment in test_comments:
    res = predict_comment(comment)
    print(f"Câu: \"{res['text']}\"")
    print(f"Chuỗi chuẩn hóa: {res['tokenized']}")
    print(f"👉 Kết quả: {res['prediction']} (Xác suất: {res['toxic_probability']})\n")

---
### 📦 BƯỚC 13: ĐÓNG GÓI TỰ ĐỘNG BÀN GIAO CHO NGƯỜI 4

In [ ]:
import shutil

zip_deliverable = "phobert_deliverables_nguoi_3.zip"
export_folder = "phobert_export"
os.makedirs(export_folder, exist_ok=True)

# Copy Checkpoint và Reports vào thư mục export
shutil.copytree(Config.OUTPUT_DIR, os.path.join(export_folder, "models", "phobert_best"), dirs_exist_ok=True)
shutil.copytree(Config.REPORT_DIR, os.path.join(export_folder, "reports", "phobert"), dirs_exist_ok=True)

# Nén thành zip
shutil.make_archive("phobert_deliverables_nguoi_3", "zip", export_folder)
print(f"🎉 ĐÃ ĐÓNG GÓI THÀNH CÔNG: {zip_deliverable}")
print("📦 Danh mục sản phẩm đã sẵn sàng gửi cho Người 4:")
print("  1. Checkpoint PhoBERT: models/phobert_best/ (weights, tokenizer, vocab, special tokens)")
print("  2. Ngưỡng tối ưu & Metrics: reports/phobert/best_metrics.json & threshold_search.csv")
print("  3. Đồ thị chất lượng cao: reports/phobert/training_and_threshold_analysis.png")
print("  4. Bảng phân tích lỗi: reports/phobert/validation_error_analysis.csv")
print("  5. Bảng so sánh baseline: reports/phobert/model_comparison.csv")

# Tải về máy tự động trên Google Colab
try:
    from google.colab import files
    files.download(zip_deliverable)
    print("📥 Đang kích hoạt tải file về máy...")
except Exception:
    print(f"ℹ️ File zip sẵn sàng tại: {os.path.abspath(zip_deliverable)}")